# 02. Treinar Splink (link_only Censo × CPF)

Profile, blocking pré-treino, treino do modelo, predict e clustering.

Usa `link_type='link_only'`: só gera pares **entre** Censo e CPF.
A coorte não entra aqui. Validação: labels em
[`03_validar_coorte.ipynb`](03_validar_coorte.ipynb); lista ouro em
[`04_validar_lista_ouro.ipynb`](04_validar_lista_ouro.ipynb).

**1ª passada:** sem nome da mãe (reserva para 2ª passada). Nomes fonéticos
com exact + Jaro-Winkler 0,98. DOB via `DateOfBirthComparison` (exact, DL≤1,
≤1 mês, ≤1 ano) — sem faixa de 10 anos.

**EDA descritiva:** [`01_analise_descritiva.ipynb`](01_analise_descritiva.ipynb).


In [1]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    OUTPUT_DIR,
    SPLINK_CLUSTERS,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    SPLINK_PREDICTIONS,
    TABELA_LIMPA,
    USE_PHONETIC_STRIP_VOWELS,
    drop_splink_temp_tables,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_tables,
)

print_paths()
con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
db_api = get_splink_db_api(con)

n_reg = con.execute(f'SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}').fetchone()[0]
duck_settings = con.execute(
    "SELECT current_setting('threads'), current_setting('memory_limit')"
).fetchone()
print(f'Registros: {n_reg:,}')
print(
    f'DuckDB: threads={duck_settings[0]}, memory_limit={duck_settings[1]} '
    f'(defaults: {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})'
)

SPLINK_ANALYSIS_SAMPLE_N = 1_000_000
analysis_table = SPLINK_INPUT_VIEW
if n_reg > SPLINK_ANALYSIS_SAMPLE_N:
    con.execute(f'''
    CREATE OR REPLACE TEMP TABLE splink_analysis_sample AS
    SELECT * FROM {SPLINK_INPUT_VIEW}
    USING SAMPLE {SPLINK_ANALYSIS_SAMPLE_N} ROWS
    ''')
    analysis_table = 'splink_analysis_sample'
    print(f'Amostra profile/blocking: {SPLINK_ANALYSIS_SAMPLE_N:,} de {n_reg:,}')


OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/censo_pessoas_2022_20260505.parquet
CENSO_CEP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/raw/censo/data_cep_uniq.csv
COHORT_DEDUP_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/capefe/dados/CohortDados/cohort_dedup.parquet
FILTRO_UF: None
FILTRO_MUNICIPIO: 2111300
USE_PHONETIC_STRIP_VOWELS: False
ANO_OBITO_CORTE: 2021
ANO_NASCIMENTO_MIN: 1900
SEXO_VALIDOS: ('M', 'F')
DUCKDB_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/probabilistico.duckdb
splink_input → registro_limpo
Registros: 2,476,718
DuckDB: threads=20, memory_limit=279.3 GiB (defaults: 20, 300GB)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Amostra profile/blocking: 1,000,000 de 2,476,718


## Sanity check - composição da base

Volume por fonte e preenchimento das colunas usadas no linkage. Blocking em
coluna muito vazia gera poucos pares candidatos.

In [2]:
from IPython.display import display

display(con.execute(f'''
SELECT origem, COUNT(*) AS n
FROM {SPLINK_INPUT_VIEW} GROUP BY 1 ORDER BY 2 DESC
''').df())

LINKAGE_COLS = [
    'primeiro_nome', 'ultimo_nome', 'nome_completo','nome_meio',
    'nome_mae', 'primeiro_nome_mae', 'nome_meio_mae', 'ultimo_nome_mae',
    'data_nascimento', 'idade', 'cep', 'sexo', 'uf',
    'nome_completo_phon','primeiro_nome_phon','nome_meio_phon','ultimo_nome_phon',
    'nome_mae_phon','primeiro_nome_mae_phon','nome_meio_mae_phon','ultimo_nome_mae_phon'
]

cols_presentes = [
    c for c in LINKAGE_COLS
    if c in set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)
]
preenchimento = ',\n    '.join(
    f"ROUND(100.0 * COUNT({c}) / COUNT(*), 1) AS pct_{c}" for c in cols_presentes
)
display(con.execute(f'''
SELECT origem, {preenchimento}
FROM {SPLINK_INPUT_VIEW} GROUP BY origem ORDER BY origem
''').df().T)


,origem,n
0,cpf,1438943
1,censo,1037775


,0,1
origem,censo,cpf
pct_primeiro_nome,95.6,100.0
pct_ultimo_nome,95.1,100.0
pct_nome_completo,95.6,100.0
pct_nome_meio,78.2,96.6
pct_nome_mae,28.9,95.8
pct_primeiro_nome_mae,28.9,95.8
pct_nome_meio_mae,23.6,89.6
pct_ultimo_nome_mae,28.9,95.8
pct_data_nascimento,83.3,97.7


## Exploração pré-modelo

Profile Splink das colunas de linkage e análise de blocking (cumulativo + maiores blocos).

In [3]:
from splink import block_on
from splink.blocking_analysis import cumulative_comparisons_to_be_scored_from_blocking_rules_chart
from splink.exploratory import profile_columns
from splink.blocking_analysis import n_largest_blocks
# Blocking rules
blocking_rules = [
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'sexo'),
    block_on('ultimo_nome_phon', 'data_nascimento'),
    block_on('primeiro_nome_phon', 'data_nascimento'),
    block_on('cep', 'ultimo_nome_phon','sexo'),
    block_on('cep', 'primeiro_nome_phon','sexo')    
]

profile_columns(
    con.execute(f'SELECT * FROM {analysis_table}').df(),
    db_api,
    column_expressions=[
        'primeiro_nome_phon', 'ultimo_nome_phon', 'nome_completo_phon',
        'cep', 'data_nascimento', 'idade',
    ],
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.VConcatChart(...)

In [4]:
# Análise de blocking só entre origens (link_only), na amostra ou na base completa.
con.execute(f"""
CREATE OR REPLACE VIEW splink_analysis_censo AS
SELECT * FROM {analysis_table} WHERE origem = 'censo'
""")
con.execute(f"""
CREATE OR REPLACE VIEW splink_analysis_cpf AS
SELECT * FROM {analysis_table} WHERE origem = 'cpf'
""")

cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=['splink_analysis_censo', 'splink_analysis_cpf'],
    blocking_rules=blocking_rules,
    db_api=db_api,
    link_type='link_only',
)


alt.Chart(...)

## Modelo Splink

Settings e `Linker` em `link_only`: duas views (Censo / CPF).
Comparisons da 1ª passada: nome (completo/primeiro/meio/último fonéticos),
DOB, idade, sexo, CEP. Sem `nome_mae*`.


In [5]:
from splink import Linker, SettingsCreator
import splink.comparison_library as cl

input_cols = set(con.execute(f'SELECT * FROM {SPLINK_INPUT_VIEW} LIMIT 0').df().columns)

comparisons = [
    cl.NameComparison('nome_completo_phon', jaro_winkler_thresholds=[0.98]).configure(term_frequency_adjustments=True),
    cl.NameComparison('primeiro_nome_phon', jaro_winkler_thresholds=[0.98]).configure(term_frequency_adjustments=True),
    cl.NameComparison('nome_meio_phon', jaro_winkler_thresholds=[0.98]).configure(term_frequency_adjustments=True),
    cl.NameComparison('ultimo_nome_phon', jaro_winkler_thresholds=[0.98]).configure(term_frequency_adjustments=True),
    cl.DateOfBirthComparison(
        'data_nascimento',
        input_is_string=True,
        datetime_thresholds=[1, 1],
        datetime_metrics=['month', 'year'],
    ),
    cl.ExactMatch('idade'),
    cl.ExactMatch('sexo').configure(term_frequency_adjustments=True),
    cl.ExactMatch('cep').configure(term_frequency_adjustments=True),
]
if USE_PHONETIC_STRIP_VOWELS and 'nome_completo_phon_sv' in input_cols:
    comparisons.append(cl.NameComparison('nome_completo_phon_sv'))

con.execute(f"""
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
""")
con.execute(f"""
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
""")
print(
    'splink_censo:', con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0],
    '| splink_cpf:', con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0],
)

settings = SettingsCreator(
    link_type='link_only',
    unique_id_column_name='unique_id',
    comparisons=comparisons,
    blocking_rules_to_generate_predictions=blocking_rules,
    retain_intermediate_calculation_columns=True,
)
linker = Linker(
    ['splink_censo', 'splink_cpf'],
    settings,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)


splink_censo: 1037775 | splink_cpf: 1438943


In [6]:
deterministic_rules = [
    block_on('nome_completo_phon', 'data_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'data_nascimento'),
]

linker.training.estimate_probability_two_random_records_match(deterministic_rules, recall=0.7)
linker.training.estimate_u_using_random_sampling(max_pairs=10_000_000)
# EM em cep+sexo: observa discordância de nome e DOB (não força nome igual no bloco).
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('cep', 'sexo'),
    estimate_without_term_frequencies=True,
)


Probability two random records match is estimated to be  4.5e-07.
This means that amongst all possible pairwise record comparisons, one in 2,221,840.61 are expected to match.  With 1,493,299,071,825 total possible comparisons, we expect a total of around 672,100.00 matching pairs
----- Estimating u probabilities using random sampling -----


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

u probability not trained for data_nascimento - Abs date difference <= 1 year (comparison vector value: 1). This usually means the comparison level was never observed in the training data.

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - nome_completo_phon (no m values are trained).
    - primeiro_nome_phon (no m values are trained).
    - nome_meio_phon (no m values are trained).
    - ultimo_nome_phon (no m values are trained).
    - nome_mae_phon (no m values are trained).
    - primeiro_nome_mae_phon (no m values are trained).
    - nome_meio_mae_phon (no m values are trained).
    - ultimo_nome_mae_phon (no m values are trained).
    - data_nascimento (some u values are not trained, no m values are trained).
    - idade (no m values are trained).
    - sexo (no m values are trained).
    - cep (no m values are trained).


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."data_nascimento" = r."data_nascimento"

Parameter estimates will be made for the following comparison(s):
    - nome_completo_phon
    - primeiro_nome_phon
    - nome_meio_phon
    - ultimo_nome_phon
    - nome_mae_phon
    - primeiro_nome_mae_phon
    - nome_meio_mae_phon
    - ultimo_nome_mae_phon
    - idade
    - sexo
    - cep

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - data_nascimento


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Iteration 1: Largest change in params was -0.637 in the m_probability of cep, level `Exact match on cep`
Iteration 2: Largest change in params was 0.141 in the m_probability of nome_completo_phon, level `All other comparisons`
Iteration 3: Largest change in params was 0.206 in the m_probability of nome_completo_phon, level `All other comparisons`
Iteration 4: Largest change in params was 0.259 in the m_probability of primeiro_nome_phon, level `All other comparisons`
Iteration 5: Largest change in params was -0.34 in the m_probability of primeiro_nome_phon, level `Exact match on primeiro_nome_phon`
Iteration 6: Largest change in params was 0.399 in the m_probability of primeiro_nome_mae_phon, level `All other comparisons`
Iteration 7: Largest change in params was 0.205 in probability_two_random_records_match
Iteration 8: Largest change in params was 0.0179 in probability_two_random_records_match
Iteration 9: Largest change in params was 0.00815 in the m_probability of idade, level `All

<EMTrainingSession, blocking on l."data_nascimento" = r."data_nascimento", deactivating comparisons data_nascimento>

In [7]:
# EM em primeiro_nome_phon + DOB: observa m de sobrenome, completo, cep, idade.
linker.training.estimate_parameters_using_expectation_maximisation(
    block_on('primeiro_nome_phon', 'data_nascimento'),
    estimate_without_term_frequencies=True,
)



----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."primeiro_nome" = r."primeiro_nome") AND (l."ultimo_nome" = r."ultimo_nome") AND (l."cep" = r."cep")

Parameter estimates will be made for the following comparison(s):
    - nome_completo_phon
    - primeiro_nome_phon
    - nome_meio_phon
    - ultimo_nome_phon
    - nome_mae_phon
    - primeiro_nome_mae_phon
    - nome_meio_mae_phon
    - ultimo_nome_mae_phon
    - data_nascimento
    - idade
    - sexo

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
    - cep

Level All other comparisons on comparison primeiro_nome_phon not observed in dataset, unable to train m value

Level All other comparisons on comparison ultimo_nome_phon not observed in dataset, unable to train m value

Level Abs date difference <= 1 year on comparison data_nascimento not observed in dataset, unable to train m value

Iteration 1: Largest change

<EMTrainingSession, blocking on (l."primeiro_nome" = r."primeiro_nome") AND (l."ultimo_nome" = r."ultimo_nome") AND (l."cep" = r."cep"), deactivating comparisons cep>

In [8]:
from config import MODELS_DIR

SPLINK_MODEL_JSON.parent.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
linker.misc.save_model_to_json(str(SPLINK_MODEL_JSON), overwrite=True)
linker.misc.save_model_to_json(str(MODELS_DIR / 'splink_model.json'), overwrite=True)
print('Modelo salvo:', SPLINK_MODEL_JSON)
print('Cópia em models/:', MODELS_DIR / 'splink_model.json')


Modelo salvo: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_model.json


## Visualização pós-treino

Match weights e registros difíceis de linkar (unlinkables).

In [9]:
linker.visualisations.match_weights_chart()


/opt/venvs/singed/jhub/lib64/python3.9/site-packages/altair/vegalite/v6/api.py:4124: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  return _tp.from_dict(dct, validate=validate)


alt.VConcatChart(...)

In [10]:
linker.evaluation.unlinkables_chart()

alt.LayerChart(...)

## Predict + clustering

`predict(0.5)` só gera candidatos. Clustering e validação usam **0,95**.


In [11]:
df_predict = linker.inference.predict(threshold_match_probability=0.5)
df_predictions = df_predict.as_pandas_dataframe()
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=0.95,
)
df_clusters = clusters.as_pandas_dataframe()
print('Pares:', len(df_predictions), 'Clusters:', df_clusters['cluster_id'].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Blocking time: 24.30 seconds


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Predict time: 105.51 seconds

 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'data_nascimento':
    m values not fully trained
Comparison: 'data_nascimento':
    u values not fully trained


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Completed iteration 2, num edges remaining to process: 112310
Completed iteration 3, num edges remaining to process: 49034
Completed iteration 4, num edges remaining to process: 25428
Completed iteration 5, num edges remaining to process: 13180
Completed iteration 6, num edges remaining to process: 6460
Completed iteration 7, num edges remaining to process: 3202
Completed iteration 8, num edges remaining to process: 1674
Completed iteration 9, num edges remaining to process: 676
Completed iteration 10, num edges remaining to process: 226
Completed iteration 11, num edges remaining to process: 86
Completed iteration 12, num edges remaining to process: 10
Completed iteration 13, num edges remaining to process: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pares: 2486120 Clusters: 1522651


## Waterfall — amostra de pares

Contribuição de cada comparação ao match weight (5 pares).

In [20]:
records_to_plot = df_predictions.tail(5).to_dict(orient="records")
linker.visualisations.waterfall_chart(records_to_plot, filter_nulls=False)


#records_sample = df_predictions.to_dict(orient='records')
#Linker.visualisations.waterfall_chart(records_sample, filter_nulls=False)

alt.LayerChart(...)

In [13]:
pd.set_option('display.max_columns', None)
#df_predictions.head(5)
#df_predictions['unique_id_l'].str.contains("censo").head(5)
df_predictions[(df_predictions['unique_id_l'].str.contains("censo")) & df_predictions['unique_id_r'].str.contains("cpf")].head(5)




,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,unique_id_r,nome_completo_phon_l,nome_completo_phon_r,gamma_nome_completo_phon,tf_nome_completo_phon_l,tf_nome_completo_phon_r,bf_nome_completo_phon,bf_tf_adj_nome_completo_phon,primeiro_nome_phon_l,primeiro_nome_phon_r,gamma_primeiro_nome_phon,tf_primeiro_nome_phon_l,tf_primeiro_nome_phon_r,bf_primeiro_nome_phon,bf_tf_adj_primeiro_nome_phon,nome_meio_phon_l,nome_meio_phon_r,gamma_nome_meio_phon,tf_nome_meio_phon_l,tf_nome_meio_phon_r,bf_nome_meio_phon,bf_tf_adj_nome_meio_phon,ultimo_nome_phon_l,ultimo_nome_phon_r,gamma_ultimo_nome_phon,tf_ultimo_nome_phon_l,tf_ultimo_nome_phon_r,bf_ultimo_nome_phon,bf_tf_adj_ultimo_nome_phon,nome_mae_phon_l,nome_mae_phon_r,gamma_nome_mae_phon,tf_nome_mae_phon_l,tf_nome_mae_phon_r,bf_nome_mae_phon,bf_tf_adj_nome_mae_phon,primeiro_nome_mae_phon_l,primeiro_nome_mae_phon_r,gamma_primeiro_nome_mae_phon,tf_primeiro_nome_mae_phon_l,tf_primeiro_nome_mae_phon_r,bf_primeiro_nome_mae_phon,bf_tf_adj_primeiro_nome_mae_phon,nome_meio_mae_phon_l,nome_meio_mae_phon_r,gamma_nome_meio_mae_phon,tf_nome_meio_mae_phon_l,tf_nome_meio_mae_phon_r,bf_nome_meio_mae_phon,bf_tf_adj_nome_meio_mae_phon,ultimo_nome_mae_phon_l,ultimo_nome_mae_phon_r,gamma_ultimo_nome_mae_phon,tf_ultimo_nome_mae_phon_l,tf_ultimo_nome_mae_phon_r,bf_ultimo_nome_mae_phon,bf_tf_adj_ultimo_nome_mae_phon,data_nascimento_l,data_nascimento_r,gamma_data_nascimento,bf_data_nascimento,idade_l,idade_r,gamma_idade,bf_idade,sexo_l,sexo_r,gamma_sexo,tf_sexo_l,tf_sexo_r,bf_sexo,bf_tf_adj_sexo,cep_l,cep_r,gamma_cep,tf_cep_l,tf_cep_r,bf_cep,bf_tf_adj_cep,match_key
0,48.217157,1.000000,censo,cpf,censo_2111300050010390060010000137000001000000005,cpf_01902355350,VANDERSON MARSELO OLIVEIRA REGO,VANDERSON MARSELO OLIVEIRA REGO,2,8.227033e-07,8.227033e-07,235343.482665,1.060004,VANDERSON,VANDERSON,1,0.001143,0.001143,57.669741,7.847426,MARSELO OLIVEIRA,MARSELO OLIVEIRA,1,0.000012,0.000012,54.591986,340.055825,REGO,REGO,1,0.001222,0.001222,22.406348,19.087782,None,RAIMUNDA OLIVEIRA REGO,-1,NaN,1.191675e-06,1.0,1.0,None,RAIMUNDA,-1,NaN,0.024646,1.0,1.0,None,OLIVEIRA,-1,NaN,0.008943,1.0,1.0,None,REGO,-1,NaN,0.001323,1.0,1.0,1986-01-15,1986-01-15,5,11461.069111,36,36,1,45.522390,M,M,1,0.480315,0.480315,1.501339,1.042757,65058241,65058063,0,0.000149,0.000067,0.992934,1.0,0
1,36.950312,1.000000,censo,cpf,censo_2111300050010390070010000159000001000000002,cpf_63960419368,RUBENIL SANTOS KOELO,RUBENIL SANTOS KOELO,2,8.227033e-07,8.227033e-07,235343.482665,1.060004,RUBENIL,RUBENIL,1,0.000002,0.000002,57.669741,3634.666184,SANTOS,SANTOS,1,0.028395,0.028395,54.591986,0.141472,KOELO,KOELO,1,0.005469,0.005469,22.406348,4.265547,None,SESILIA PEDROSA SANTOS,-1,NaN,1.191675e-06,1.0,1.0,None,SESILIA,-1,NaN,0.001133,1.0,1.0,None,PEDROSA,-1,NaN,0.000124,1.0,1.0,None,SANTOS,-1,NaN,0.076886,1.0,1.0,1977-10-19,1977-10-19,5,11461.069111,44,45,0,0.429031,M,M,1,0.480315,0.480315,1.501339,1.042757,65058241,65058063,0,0.000149,0.000067,0.992934,1.0,0
2,34.020914,1.000000,censo,cpf,censo_2111300050010390070010000161000001000000004,cpf_61294305310,LUSIMARA SANTOS SILVA,LUSIMARA SANTOS SILVA,2,1.645407e-06,1.645407e-06,235343.482665,0.530002,LUSIMARA,LUSIMARA,1,0.000050,0.000050,57.669741,180.231381,SANTOS,SANTOS,1,0.028395,0.028395,54.591986,0.141472,SILVA,SILVA,1,0.101296,0.101296,22.406348,0.230292,None,MARIA LOURDES SILVA SANTOS,-1,NaN,5.005035e-05,1.0,1.0,None,MARIA,-1,NaN,0.230407,1.0,1.0,None,LOURDES SILVA,-1,NaN,0.000514,1.0,1.0,None,SANTOS,-1,NaN,0.076886,1.0,1.0,1996-05-16,1996-05-16,5,11461.069111,26,26,1,45.522390,F,F,1,0.519685,0.519685,1.501339,0.963762,65058241,65000000,0,0.000149,0.118874,0.992934,1.0,0
3,55.964659,1.000000,censo,cpf,censo_2111300050010400010040000048000001000000001,cpf_62492829391,DIANA MARSIA SAMPAIO MATEUS,DIANA MARSIA SAMPAIO MATEUS,2,1.234055e-06,1.234055e-06,235343.482665,0.706669,DIANA,DIANA,1,0.000643,0.000643,57.669741,13.952653,MARSIA SAMPAIO,MARSIA SAMPAIO,1,0.000001,0.000001,

## Cluster studio

Dashboard interativo para inspecionar clusters (amostra por tamanho).

In [14]:
from IPython.display import IFrame, display

CLUSTER_STUDIO_HTML = OUTPUT_DIR / 'dashboards' / 'cluster_studio.html'
CLUSTER_STUDIO_HTML.parent.mkdir(parents=True, exist_ok=True)

linker.visualisations.cluster_studio_dashboard(
    df_predict,
    clusters,
    str(CLUSTER_STUDIO_HTML),
    sampling_method='by_cluster_size',
    overwrite=True,
)
print('Dashboard:', CLUSTER_STUDIO_HTML)
display(IFrame(src=str(CLUSTER_STUDIO_HTML), width='100%', height=1200))

Dashboard: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/dashboards/cluster_studio.html


## Diagnóstico dos scores (sem rótulos)

Distribuição de match weight e tamanho de cluster. Labels Splink no NB03;
funil da lista ouro no NB04. Cheque os match weights: discordar nome/DOB
deve penalizar vários bits, não ≈ 0.


In [15]:
display(df_predictions['match_probability'].describe())
faixas = pd.cut(df_predictions['match_weight'], bins=20)
display(
    df_predictions.groupby(faixas, observed=True)
    .size()
    .rename('n_pares')
    .to_frame()
)

count    2.486120e+06
mean     8.722246e-01
std      1.657603e-01
min      5.000131e-01
25%      7.670333e-01
50%      9.732455e-01
75%      9.999908e-01
max      1.000000e+00
Name: match_probability, dtype: float64

,n_pares
match_weight,
"(-0.133, 6.659]",1402909
"(6.659, 13.318]",386579
"(13.318, 19.977]",120279
"(19.977, 26.636]",78928
"(26.636, 33.295]",90259
"(33.295, 39.954]",108019
"(39.954, 46.614]",95347
"(46.614, 53.273]",57945
"(53.273, 59.932]",32450


In [16]:
tamanhos = df_clusters.groupby('cluster_id').size()
print(f'Clusters: {tamanhos.size:,} | maior: {tamanhos.max():,} | singletons: {(tamanhos == 1).sum():,}')
display(
    tamanhos.value_counts().sort_index().head(20)
    .rename('n_clusters').to_frame().rename_axis('tamanho')
)

# Clusters grandes demais indicam blocking/threshold frouxo — inspecionar antes do NB03.
display(tamanhos.sort_values(ascending=False).head(10).rename('tamanho').to_frame())

Clusters: 1,522,651 | maior: 41,692 | singletons: 946,728


,n_clusters
tamanho,
1,946728
2,464528
3,50267
4,28933
5,10099
6,6831
7,3715
8,2627
9,1763


,tamanho
cluster_id,
censo-__-censo_2111300050000010010020000034000002000000003,41692
censo-__-censo_2111300050000170010110000106000001000000003,1084
censo-__-censo_2111300050000240150080000375000001000000002,524
censo-__-censo_2111300050000230020020000033000001000000002,402
censo-__-censo_2111300050000090090020000199000001000000006,362
censo-__-censo_2111300050000080100020000303000002000000001,357
censo-__-censo_2111300050000140180040000239000001000000001,351
censo-__-censo_2111300050000010100030000267000001000000004,343
censo-__-censo_2111300050000130190020000313000001000000002,318


In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_predictions.to_parquet(SPLINK_PREDICTIONS)
df_clusters.to_parquet(SPLINK_CLUSTERS)
print('Predictions:', SPLINK_PREDICTIONS)
print('Clusters:', SPLINK_CLUSTERS)

Predictions: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_predictions.parquet
Clusters: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_clusters.parquet


## Encerrar

Artefatos prontos para o [`03_validar_coorte.ipynb`](03_validar_coorte.ipynb).

In [18]:
for label, p in [
    ('modelo', SPLINK_MODEL_JSON),
    ('predictions', SPLINK_PREDICTIONS),
    ('clusters', SPLINK_CLUSTERS),
]:
    print(f'{label:12s} {"ok " if p.exists() else "FALTA"} {p}')

con.close()


modelo       ok  /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_model.json
predictions  ok  /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_predictions.parquet
clusters     ok  /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/splink_clusters.parquet


In [19]:
con.close()